In [ ]:
# PyTorch core libraries
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, random_split
from torch.utils.tensorboard import SummaryWriter

# Math utilities
import math
import numpy as np

# HuggingFace libraries
from datasets import load_dataset
from tokenizers import Tokenizer
from tokenizers.models import WordLevel
from tokenizers.trainers import WordLevelTrainer
from tokenizers.pre_tokenizers import Whitespace

# Pathlib for filesystem paths
from pathlib import Path

# Typing for type hints
from typing import Any

# Progress bar utility
from tqdm import tqdm

# Warnings management
import warnings



## **Test Our Transformer**

In [ ]:
# Dummy classes needed for the Transformer to be initialized and built.
# These would typically be fully implemented in a real Transformer.
class InputEmbeddings(nn.Module):
    def __init__(self, d_model: int, vocab_size: int) -> None:
        super().__init__()
        self.d_model = d_model
        self.vocab_size = vocab_size
        self.embedding = nn.Embedding(vocab_size, d_model)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.embedding(x) * torch.sqrt(torch.tensor(self.d_model, dtype=torch.float32))

class PositionalEncoding(nn.Module):
    def __init__(self, d_model: int, seq_len: int, dropout: float) -> None:
        super().__init__()
        self.d_model = d_model
        self.seq_len = seq_len
        self.dropout = nn.Dropout(dropout)
        # Create a positional encoding matrix
        pe = torch.zeros(seq_len, d_model)
        position = torch.arange(0, seq_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-torch.log(torch.tensor(10000.0)) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0) # (1, seq_len, d_model)
        self.register_buffer('pe', pe)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = x + (self.pe[:, :x.shape[1], :]).requires_grad_(False)
        return self.dropout(x)

class MultiHeadAttention(nn.Module):
    def __init__(self, d_model: int, h: int, dropout: float) -> None:
        super().__init__()
        self.d_model = d_model
        self.h = h
        assert d_model % h == 0, "d_model is not divisible by h"
        self.d_k = d_model // h
        self.w_q = nn.Linear(d_model, d_model)
        self.w_k = nn.Linear(d_model, d_model)
        self.w_v = nn.Linear(d_model, d_model)
        self.w_o = nn.Linear(d_model, d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, q, k, v, mask):
        # In a real implementation, attention logic would go here.
        # For a dummy, we just pass through and reshape.
        query = self.w_q(q)
        key = self.w_k(k)
        value = self.w_v(v)

        query = query.view(query.shape[0], query.shape[1], self.h, self.d_k).transpose(1, 2)
        key = key.view(key.shape[0], key.shape[1], self.h, self.d_k).transpose(1, 2)
        value = value.view(value.shape[0], value.shape[1], self.h, self.d_k).transpose(1, 2)

        # For dummy, just return a tensor of the expected shape
        # (batch_size, seq_len, d_model)
        output = torch.randn(q.shape[0], q.shape[1], self.d_model).to(q.device)
        return self.w_o(output) # Apply output linear layer

class FeedForwardNetwork(nn.Module):
    def __init__(self, d_model: int, d_ff: int, dropout: float) -> None:
        super().__init__()
        self.linear_1 = nn.Linear(d_model, d_ff)
        self.dropout = nn.Dropout(dropout)
        self.linear_2 = nn.Linear(d_ff, d_model)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.linear_2(self.dropout(torch.relu(self.linear_1(x))))

class LayerNormalization(nn.Module):
    def __init__(self, features: int, eps: float = 1e-6) -> None:
        super().__init__()
        self.eps = eps
        self.alpha = nn.Parameter(torch.ones(features))
        self.bias = nn.Parameter(torch.zeros(features))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        mean = x.mean(dim=-1, keepdim=True)
        std = x.std(dim=-1, keepdim=True)
        return self.alpha * (x - mean) / (std + self.eps) + self.bias

class EncoderBlock(nn.Module):
    def __init__(self, self_attention: MultiHeadAttention, feed_forward: FeedForwardNetwork, dropout: float = 0.1) -> None:
        super().__init__()
        self.self_attention = self_attention
        self.feed_forward = feed_forward
        self.norm1 = LayerNormalization(self_attention.d_model)
        self.norm2 = LayerNormalization(feed_forward.linear_2.out_features)
        self.dropout1 = nn.Dropout(dropout)
        self.dropout2 = nn.Dropout(dropout)

    def forward(self, x: torch.Tensor, src_mask: torch.Tensor) -> torch.Tensor:
        x = x + self.dropout1(self.self_attention(x, x, x, src_mask))
        x = self.norm1(x)
        x = x + self.dropout2(self.feed_forward(x))
        x = self.norm2(x)
        return x

class Encoder(nn.Module):
    def __init__(self, layers: nn.ModuleList) -> None:
        super().__init__()
        self.layers = layers
        self.norm = LayerNormalization(layers[0].self_attention.d_model) # Assuming d_model is consistent

    def forward(self, x: torch.Tensor, mask: torch.Tensor) -> torch.Tensor:
        for layer in self.layers:
            x = layer(x, mask)
        return self.norm(x)

class DecoderBlock(nn.Module):
    def __init__(self, self_attention: MultiHeadAttention, cross_attention: MultiHeadAttention, feed_forward: FeedForwardNetwork, dropout: float = 0.1) -> None:
        super().__init__()
        self.self_attention = self_attention
        self.cross_attention = cross_attention
        self.feed_forward = feed_forward
        self.norm1 = LayerNormalization(self_attention.d_model)
        self.norm2 = LayerNormalization(cross_attention.d_model)
        self.norm3 = LayerNormalization(feed_forward.linear_2.out_features)
        self.dropout1 = nn.Dropout(dropout)
        self.dropout2 = nn.Dropout(dropout)
        self.dropout3 = nn.Dropout(dropout)

    def forward(self, x: torch.Tensor, encoder_output: torch.Tensor, src_mask: torch.Tensor, tgt_mask: torch.Tensor) -> torch.Tensor:
        x = x + self.dropout1(self.self_attention(x, x, x, tgt_mask))
        x = self.norm1(x)
        x = x + self.dropout2(self.cross_attention(x, encoder_output, encoder_output, src_mask))
        x = self.norm2(x)
        x = x + self.dropout3(self.feed_forward(x))
        x = self.norm3(x)
        return x

class Decoder(nn.Module):
    def __init__(self, layers: nn.ModuleList) -> None:
        super().__init__()
        self.layers = layers
        self.norm = LayerNormalization(layers[0].self_attention.d_model) # Assuming d_model is consistent

    def forward(self, x: torch.Tensor, encoder_output: torch.Tensor, src_mask: torch.Tensor, tgt_mask: torch.Tensor) -> torch.Tensor:
        for layer in self.layers:
            x = layer(x, encoder_output, src_mask, tgt_mask)
        return self.norm(x)

class ProjectionLayer(nn.Module):
    def __init__(self, d_model: int, vocab_size: int) -> None:
        super().__init__()
        self.proj = nn.Linear(d_model, vocab_size)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # (batch_size, seq_len, d_model) -> (batch_size, seq_len, vocab_size)
        return torch.log_softmax(self.proj(x), dim=-1)

class Transformer(nn.Module):
    def __init__(
        self,
        encoder: Encoder,
        decoder: Decoder,
        encoder_embedding: InputEmbeddings,
        decoder_embedding: InputEmbeddings,
        encoder_positional_encoding: PositionalEncoding,
        decoder_positional_encoding: PositionalEncoding,
        projection_layer: ProjectionLayer
    ) -> None:
        super().__init__()

        # Encoder and Decoder blocks
        self.encoder_block = encoder
        self.decoder_block = decoder

        # Input Embeddings
        self.encoder_embedding = encoder_embedding
        self.decoder_embedding = decoder_embedding

        # Positional Encodings
        self.encoder_positional_encoding = encoder_positional_encoding
        self.decoder_positional_encoding = decoder_positional_encoding

        # Output Projection Layer
        self.projection_layer = projection_layer

    def encode(
        self,
        encoder_input: torch.Tensor,
        encoder_mask: torch.Tensor
    ) -> torch.Tensor:
        # Step 1: Convert encoder input tokens into embeddings
        encoder_embedded = self.encoder_embedding(encoder_input)

        # Step 2: Add positional information
        encoder_position_encoded = (
            self.encoder_positional_encoding(encoder_embedded)
        )

        # Step 3: Pass through encoder blocks
        encoder_output = self.encoder_block(
            encoder_position_encoded,
            encoder_mask
        )
        return encoder_output

    def decode(
        self,
        encoder_output: torch.Tensor,
        encoder_mask: torch.Tensor,
        decoder_input: torch.Tensor,
        decoder_mask: torch.Tensor
    ) -> torch.Tensor:
        # Step 1: Convert decoder input tokens into embeddings
        decoder_embedded = self.decoder_embedding(decoder_input)

        # Step 2: Add positional information
        decoder_position_encoded = (
            self.decoder_positional_encoding(decoder_embedded)
        )

        # Step 3: Pass through decoder blocks
        decoder_output = self.decoder_block(
            decoder_position_encoded,
            encoder_output,
            encoder_mask,
            decoder_mask
        )
        return decoder_output

    def project(
        self,
        decoder_output: torch.Tensor
    ) -> torch.Tensor:
        # Convert decoder output into vocabulary logits
        projected_output = self.projection_layer(
            decoder_output
        )
        return projected_output

In [ ]:
def build_transformer(input_vocab_size:int,output_vocab_size:int,input_seq_len:int,decoder_seq_len:int,d_model:int=512,nhead:int=8,num_encoder_layers:int=6,num_decoder_layers:int=6,dim_feedforward:int=2048,dropout:float=0.1)->Transformer:
  encoder_embedding = InputEmbeddings(d_model,input_vocab_size)
  decoder_embedding = InputEmbeddings(d_model,output_vocab_size)

  encoder_positional_encoding = PositionalEncoding(d_model,input_seq_len,dropout)
  decoder_positional_encoding = PositionalEncoding(d_model,decoder_seq_len,dropout)

  encoder_layers = []
  for _ in range(num_encoder_layers):
    encoder_self_attention = MultiHeadAttention(d_model,nhead,dropout)
    feed_forward_encoder = FeedForwardNetwork(d_model,dim_feedforward,dropout)
    encoder_block = EncoderBlock(encoder_self_attention,feed_forward_encoder,dropout)
    encoder_layers.append(encoder_block)

  decoder_layers = []
  for _ in range(num_decoder_layers):
    decoder_self_attention = MultiHeadAttention(d_model,nhead,dropout)
    encoder_decoder_attention = MultiHeadAttention(d_model,nhead,dropout)
    feed_forward_decoder = FeedForwardNetwork(d_model,dim_feedforward,dropout)
    decoder_block = DecoderBlock(decoder_self_attention,encoder_decoder_attention,feed_forward_decoder,dropout)
    decoder_layers.append(decoder_block)

  encoder = Encoder(nn.ModuleList(encoder_layers))
  decoder = Decoder(nn.ModuleList(decoder_layers))


  projection_layer = ProjectionLayer(d_model,output_vocab_size)

  transformer = Transformer(
      encoder=encoder,
      decoder=decoder,
      encoder_embedding=encoder_embedding,
      decoder_embedding=decoder_embedding,
      encoder_positional_encoding=encoder_positional_encoding,
      decoder_positional_encoding=decoder_positional_encoding,
      projection_layer=projection_layer
  )

  for p in transformer.parameters():
    if p.dim() > 1:
      nn.init.xavier_uniform_(p)

  return transformer

In [ ]:
# Define the parameters for our dummy Transformer
input_vocab_size = 100  # Number of unique words/tokens in the input language
output_vocab_size = 120 # Number of unique words/tokens in the output language
input_seq_len = 10      # Maximum length of the input sequence (e.g., 10 words)
decoder_seq_len = 12    # Maximum length of the target/output sequence (e.g., 12 words)
d_model = 64            # The dimension of the embeddings and sub-layers outputs
nhead = 4               # Number of attention heads (splits d_model into nhead chunks)
num_encoder_layers = 2  # Number of encoder blocks
num_decoder_layers = 2  # Number of decoder blocks
dim_feedforward = 128   # Dimension of the feed-forward network in each block
dropout = 0.1           # Dropout rate to prevent overfitting

# Build the transformer model with these parameters
transformer = build_transformer(
    input_vocab_size,
    output_vocab_size,
    input_seq_len,
    decoder_seq_len,
    d_model,
    nhead,
    num_encoder_layers,
    num_decoder_layers,
    dim_feedforward,
    dropout
)

print("Transformer model built successfully!")
print(transformer)

Transformer model built successfully!
Transformer(
  (encoder_block): Encoder(
    (layers): ModuleList(
      (0-1): 2 x EncoderBlock(
        (self_attention): MultiHeadAttention(
          (w_q): Linear(in_features=64, out_features=64, bias=True)
          (w_k): Linear(in_features=64, out_features=64, bias=True)
          (w_v): Linear(in_features=64, out_features=64, bias=True)
          (w_o): Linear(in_features=64, out_features=64, bias=True)
          (dropout): Dropout(p=0.1, inplace=False)
        )
        (feed_forward): FeedForwardNetwork(
          (linear_1): Linear(in_features=64, out_features=128, bias=True)
          (dropout): Dropout(p=0.1, inplace=False)
          (linear_2): Linear(in_features=128, out_features=64, bias=True)
        )
        (norm1): LayerNormalization()
        (norm2): LayerNormalization()
        (dropout1): Dropout(p=0.1, inplace=False)
        (dropout2): Dropout(p=0.1, inplace=False)
      )
    )
    (norm): LayerNormalization()
  )
  (de

In [ ]:
# Create dummy input data
batch_size = 1

# Dummy encoder input (batch_size, input_seq_len)
# These are token IDs, like words in a sentence
encoder_input = torch.randint(0, input_vocab_size, (batch_size, input_seq_len))

# Dummy decoder input (batch_size, decoder_seq_len)
# These are token IDs for the target sequence
decoder_input = torch.randint(0, output_vocab_size, (batch_size, decoder_seq_len))

# Dummy masks (batch_size, 1, seq_len) for attention mechanisms
# For simplicity, we'll create masks that don't block anything initially
encoder_mask = torch.ones(batch_size, 1, input_seq_len) # No masking for now
decoder_mask = torch.ones(batch_size, 1, decoder_seq_len) # No masking for now

print("Dummy input data created:")
print(f"Encoder Input Shape: {encoder_input.shape}")
print(f"Decoder Input Shape: {decoder_input.shape}")
print(f"Encoder Mask Shape: {encoder_mask.shape}")
print(f"Decoder Mask Shape: {decoder_mask.shape}")

Dummy input data created:
Encoder Input Shape: torch.Size([1, 10])
Decoder Input Shape: torch.Size([1, 12])
Encoder Mask Shape: torch.Size([1, 1, 10])
Decoder Mask Shape: torch.Size([1, 1, 12])



#### **1. Encoder Input: Embedding and Positional Encoding**

Before the encoder can do its magic, our raw input tokens (`encoder_input`) are transformed into meaningful numerical representations called 'embeddings'. Then, 'positional encodings' are added to these embeddings to give the Transformer a sense of word order, as it doesn't inherently understand sequence.


In [ ]:
print("--- Encoder Path ---")

# Step 1: Encoder Embedding
encoder_embedded = transformer.encoder_embedding(encoder_input)
print(f"Encoder Embedding Shape: {encoder_embedded.shape} (batch, seq_len, d_model)")

# Step 2: Encoder Positional Encoding
encoder_position_encoded = transformer.encoder_positional_encoding(encoder_embedded)
print(f"Encoder Positional Encoded Shape: {encoder_position_encoded.shape} (batch, seq_len, d_model)")

--- Encoder Path ---
Encoder Embedding Shape: torch.Size([1, 10, 64]) (batch, seq_len, d_model)
Encoder Positional Encoded Shape: torch.Size([1, 10, 64]) (batch, seq_len, d_model)


#### **2. Encoder Output: Understanding the Input Sentence**

The positional-encoded input then goes through the Encoder blocks. Each block has a Self-Attention layer (to understand relationships within the input sentence) and a Feed-Forward Network. The final output of the encoder is a rich, contextual representation of the entire input sentence.

In [ ]:
# Step 3: Encoder Blocks (producing the encoder's 'understanding' of the input)
encoder_output = transformer.encode(encoder_input, encoder_mask)
print(f"Encoder Output Shape: {encoder_output.shape} (batch, seq_len, d_model)")

Encoder Output Shape: torch.Size([1, 10, 64]) (batch, seq_len, d_model)


#### **3. Decoder Input: Embedding and Positional Encoding**

Similar to the encoder, the decoder's input (the target sequence) is first embedded and then positional encodings are added. This initial input helps the decoder start generating the output.

In [ ]:
print("\n--- Decoder Path ---")

# Step 1: Decoder Embedding
decoder_embedded = transformer.decoder_embedding(decoder_input)
print(f"Decoder Embedding Shape: {decoder_embedded.shape} (batch, seq_len, d_model)")

# Step 2: Decoder Positional Encoding
decoder_position_encoded = transformer.decoder_positional_encoding(decoder_embedded)
print(f"Decoder Positional Encoded Shape: {decoder_position_encoded.shape} (batch, seq_len, d_model)")


--- Decoder Path ---
Decoder Embedding Shape: torch.Size([1, 12, 64]) (batch, seq_len, d_model)
Decoder Positional Encoded Shape: torch.Size([1, 12, 64]) (batch, seq_len, d_model)


#### **4. Decoder Output: Generating the Translated Sentence**

The positional-encoded decoder input, along with the `encoder_output`, passes through the Decoder blocks. Each decoder block has three main parts:
1.  **Self-Attention**: Helps the decoder understand the context of the words it has *already* generated in the output sequence.
2.  **Cross-Attention**: This is where the decoder 'looks at' the encoder's understanding of the input sentence (`encoder_output`) to decide what word to generate next.
3.  **Feed-Forward Network**: Processes the combined information.

The final `decoder_output` contains contextual representations of the words the Transformer is trying to generate.

In [ ]:
# Step 3: Decoder Blocks (generating the target sequence)
decoder_output = transformer.decode(
    encoder_output,
    encoder_mask,
    decoder_input,
    decoder_mask
)
print(f"Decoder Output Shape: {decoder_output.shape} (batch, seq_len, d_model)")

Decoder Output Shape: torch.Size([1, 12, 64]) (batch, seq_len, d_model)


#### **5. Projection Layer Output: The Final Word Choice**

Finally, the `decoder_output` goes through a 'Projection Layer'. This layer takes the complex numerical representation from the decoder and projects it back into the size of our output vocabulary. Essentially, it assigns a probability or 'score' to every possible word in our target language for each position in the output sequence. The word with the highest score is then chosen as the next word in the translation.


In [ ]:
# Step 4: Projection Layer (converting decoder output into probabilities for each word in the vocabulary)
projected_output = transformer.project(decoder_output)
print(f"Projected Output Shape: {projected_output.shape} (batch, seq_len, output_vocab_size)")

# To get the predicted words, we would typically take the argmax of the projected_output
# For example, for the first word in the first batch:
predicted_token_ids = torch.argmax(projected_output, dim=-1)
print(f"Example Predicted Token IDs (first batch): {predicted_token_ids[0]}")

print("\nThis completes the forward pass through the Transformer!")

Projected Output Shape: torch.Size([1, 12, 120]) (batch, seq_len, output_vocab_size)
Example Predicted Token IDs (first batch): tensor([  4,  60,   9,   5,   3, 119,  12,  86,  60,  42,  92,   0])

This completes the forward pass through the Transformer!
